# Silver Layer: Creation & Validation

This notebook creates the Silver layer tables for Green Taxi and Taxi Zones from Bronze, applying data quality rules, preserving provenance columns, and running comprehensive validation checks.

Source: `ftw-week-08`.`01_bronze`  →  Target: `ftw-week-08`.`02_silver`

## 1. Green Taxi Silver

Create the Silver `green_taxi` table from Bronze, applying data quality rules:
- Trip distance > 1000 is set to NULL and flagged
- Pickup/dropoff datetimes outside March–May 2026 are flagged
- `_rescued_data` and `ehail_fee` are dropped (100% NULL)
- Provenance columns are preserved

In [0]:
%sql
CREATE OR REPLACE TABLE `ftw-week-08`.`02_silver`.`green_taxi`
USING DELTA
AS
SELECT
  -- Primary identifiers
  VendorID,
  
  -- Timestamps (keep as-is, flagged separately)
  lpep_pickup_datetime,
  lpep_dropoff_datetime,
  
  -- Trip identifiers and codes
  store_and_fwd_flag,
  RatecodeID,
  PULocationID,
  DOLocationID,
  passenger_count,
  
  -- Trip distance: NULL if > 1000, otherwise keep original value
  CASE 
    WHEN trip_distance > 1000 THEN NULL
    ELSE trip_distance
  END AS trip_distance,
  
  -- Financial columns (keep all values including negatives)
  fare_amount,
  extra,
  mta_tax,
  tip_amount,
  tolls_amount,
  improvement_surcharge,
  total_amount,
  
  -- Payment and trip type
  payment_type,
  trip_type,
  
  -- Surcharges
  congestion_surcharge,
  cbd_congestion_fee,
  
  -- Data Quality Flags
  CASE 
    WHEN trip_distance > 1000 THEN TRUE
    ELSE FALSE
  END AS dq_invalid_trip_distance,
  
  CASE
    WHEN lpep_pickup_datetime < TIMESTAMP '2026-03-01 00:00:00'
      OR lpep_pickup_datetime >= TIMESTAMP '2026-06-01 00:00:00'
      OR lpep_dropoff_datetime < TIMESTAMP '2026-03-01 00:00:00'
      OR lpep_dropoff_datetime >= TIMESTAMP '2026-06-01 00:00:00'
    THEN TRUE
    ELSE FALSE
  END AS dq_out_of_range_datetime,
  
  -- Provenance columns
  source_system,
  source_file,
  batch_id,
  ingested_at
  
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
-- Note: _rescued_data and ehail_fee are dropped (100% NULL)

## 2. Green Taxi Validation

Validation checks for the Silver `green_taxi` table:
- Row count and source file count
- Invalid trip-distance checks and nullification verification
- Out-of-range datetime checks
- Systematic NULL pattern checks
- Sample flagged records
- Schema/column verification

In [0]:
%sql
-- Verify the Silver table was created successfully
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT source_file) AS source_files,
  SUM(CASE WHEN dq_invalid_trip_distance THEN 1 ELSE 0 END) AS rows_with_invalid_trip_distance,
  SUM(CASE WHEN dq_out_of_range_datetime THEN 1 ELSE 0 END) AS rows_with_out_of_range_datetime,
  SUM(CASE WHEN trip_distance IS NULL AND dq_invalid_trip_distance THEN 1 ELSE 0 END) AS trip_distance_nullified,
  SUM(CASE WHEN RatecodeID IS NULL 
           AND congestion_surcharge IS NULL 
           AND passenger_count IS NULL 
           AND payment_type IS NULL 
           AND store_and_fwd_flag IS NULL 
           AND trip_type IS NULL THEN 1 ELSE 0 END) AS systematic_null_pattern_rows
FROM `ftw-week-08`.`02_silver`.`green_taxi`

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT source_file) AS source_files,

    -- Extreme trip distance handling
    COUNT_IF(trip_distance > 1000) AS remaining_invalid_trip_distance,
    COUNT_IF(dq_invalid_trip_distance = TRUE) AS flagged_invalid_trip_distance,
    COUNT_IF(
        dq_invalid_trip_distance = TRUE
        AND trip_distance IS NULL
    ) AS trip_distance_nullified,

    -- Out-of-range datetime handling
    COUNT_IF(dq_out_of_range_datetime = TRUE) AS flagged_out_of_range_datetime,

    -- Systematic NULL pattern preservation
    COUNT_IF(
        RatecodeID IS NULL
        AND congestion_surcharge IS NULL
        AND passenger_count IS NULL
        AND payment_type IS NULL
        AND store_and_fwd_flag IS NULL
        AND trip_type IS NULL
    ) AS systematic_null_pattern_rows

FROM `ftw-week-08`.`02_silver`.`green_taxi`;

In [0]:
%sql
-- Show sample rows with data quality issues flagged
SELECT 
  VendorID,
  lpep_pickup_datetime,
  lpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  total_amount,
  dq_invalid_trip_distance,
  dq_out_of_range_datetime,
  passenger_count,
  RatecodeID,
  payment_type
FROM `ftw-week-08`.`02_silver`.`green_taxi`
WHERE dq_invalid_trip_distance = TRUE OR dq_out_of_range_datetime = TRUE
LIMIT 10

In [0]:
%sql
-- Verify all required columns are present and _rescued_data and ehail_fee are dropped
DESCRIBE `ftw-week-08`.`02_silver`.`green_taxi`

## 3. Taxi Zones Silver

Create the Silver `taxi_zones` table from Bronze with type casting for all columns.

In [0]:
%sql
CREATE OR REPLACE TABLE `ftw-week-08`.`02_silver`.`taxi_zones`
USING DELTA
AS
SELECT
    CAST(LocationID AS INT) AS LocationID,
    CAST(Borough AS STRING) AS Borough,
    CAST(Zone AS STRING) AS Zone,
    CAST(service_zone AS STRING) AS service_zone
FROM `ftw-week-08`.`01_bronze`.`taxi_zones`

## 4. Taxi Zones Validation

Validation checks for the Silver `taxi_zones` table:
- Source vs Silver row count
- NULL checks on all columns
- Duplicate LocationID check
- LocationID range and distinct-count check
- Pickup/dropoff orphan checks against Bronze Green Taxi
- Final PASS/FAIL validation summary

In [0]:
%sql
-- Compare source and Silver row counts
SELECT
    'Source (Bronze)' AS table_name,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`01_bronze`.`taxi_zones`

UNION ALL

SELECT
    'Target (Silver)' AS table_name,
    COUNT(*) AS row_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`

In [0]:
%sql
-- Check for NULL values in all columns
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(LocationID IS NULL) AS null_location_id,
    COUNT_IF(Borough IS NULL) AS null_borough,
    COUNT_IF(Zone IS NULL) AS null_zone,
    COUNT_IF(service_zone IS NULL) AS null_service_zone
FROM `ftw-week-08`.`02_silver`.`taxi_zones`

In [0]:
%sql
-- Check for duplicate LocationIDs
SELECT
    LocationID,
    COUNT(*) AS record_count
FROM `ftw-week-08`.`02_silver`.`taxi_zones`
GROUP BY LocationID
HAVING COUNT(*) > 1

In [0]:
%sql
-- Check LocationID range and distinct count
SELECT
    MIN(LocationID) AS min_location_id,
    MAX(LocationID) AS max_location_id,
    COUNT(DISTINCT LocationID) AS distinct_location_ids
FROM `ftw-week-08`.`02_silver`.`taxi_zones`

In [0]:
%sql
-- Check for orphan pickup and dropoff location IDs
SELECT
    'pickup' AS location_type,
    COUNT(*) AS orphan_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi` g
LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z
    ON g.PULocationID = z.LocationID
WHERE g.PULocationID IS NOT NULL
  AND z.LocationID IS NULL

UNION ALL

SELECT
    'dropoff' AS location_type,
    COUNT(*) AS orphan_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi` g
LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z
    ON g.DOLocationID = z.LocationID
WHERE g.DOLocationID IS NOT NULL
  AND z.LocationID IS NULL

In [0]:
%sql
-- Final validation summary with Pass/Fail status
WITH validation_results AS (
    SELECT
        (SELECT COUNT(*) FROM `ftw-week-08`.`01_bronze`.`taxi_zones`) AS source_count,
        (SELECT COUNT(*) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS silver_count,
        (SELECT COUNT_IF(LocationID IS NULL) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_location_id,
        (SELECT COUNT_IF(Borough IS NULL) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_borough,
        (SELECT COUNT_IF(Zone IS NULL) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_zone,
        (SELECT COUNT_IF(service_zone IS NULL) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS null_service_zone,
        (SELECT COUNT(*) FROM (
            SELECT LocationID, COUNT(*) AS cnt
            FROM `ftw-week-08`.`02_silver`.`taxi_zones`
            GROUP BY LocationID
            HAVING COUNT(*) > 1
        )) AS duplicate_location_id,
        (SELECT COUNT(DISTINCT LocationID) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS distinct_location_ids,
        (SELECT MIN(LocationID) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS min_location_id,
        (SELECT MAX(LocationID) FROM `ftw-week-08`.`02_silver`.`taxi_zones`) AS max_location_id,
        (SELECT COUNT(*) FROM `ftw-week-08`.`01_bronze`.`green_taxi` g
         LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z ON g.PULocationID = z.LocationID
         WHERE g.PULocationID IS NOT NULL AND z.LocationID IS NULL) AS orphan_pickup,
        (SELECT COUNT(*) FROM `ftw-week-08`.`01_bronze`.`green_taxi` g
         LEFT JOIN `ftw-week-08`.`02_silver`.`taxi_zones` z ON g.DOLocationID = z.LocationID
         WHERE g.DOLocationID IS NOT NULL AND z.LocationID IS NULL) AS orphan_dropoff
)
SELECT
    check_name,
    expected,
    actual,
    CASE WHEN expected = actual THEN 'PASS' ELSE 'FAIL' END AS status
FROM (
    SELECT 'Source row count' AS check_name, 265 AS expected, source_count AS actual FROM validation_results
    UNION ALL
    SELECT 'Silver row count', 265, silver_count FROM validation_results
    UNION ALL
    SELECT 'NULL LocationID', 0, null_location_id FROM validation_results
    UNION ALL
    SELECT 'NULL Borough', 0, null_borough FROM validation_results
    UNION ALL
    SELECT 'NULL Zone', 0, null_zone FROM validation_results
    UNION ALL
    SELECT 'NULL service_zone', 0, null_service_zone FROM validation_results
    UNION ALL
    SELECT 'Duplicate LocationID', 0, duplicate_location_id FROM validation_results
    UNION ALL
    SELECT 'Distinct LocationID', 265, distinct_location_ids FROM validation_results
    UNION ALL
    SELECT 'Minimum LocationID', 1, min_location_id FROM validation_results
    UNION ALL
    SELECT 'Maximum LocationID', 265, max_location_id FROM validation_results
    UNION ALL
    SELECT 'Orphan pickup IDs', 0, orphan_pickup FROM validation_results
    UNION ALL
    SELECT 'Orphan dropoff IDs', 0, orphan_dropoff FROM validation_results
)
ORDER BY
    CASE check_name
        WHEN 'Source row count' THEN 1
        WHEN 'Silver row count' THEN 2
        WHEN 'NULL LocationID' THEN 3
        WHEN 'NULL Borough' THEN 4
        WHEN 'NULL Zone' THEN 5
        WHEN 'NULL service_zone' THEN 6
        WHEN 'Duplicate LocationID' THEN 7
        WHEN 'Distinct LocationID' THEN 8
        WHEN 'Minimum LocationID' THEN 9
        WHEN 'Maximum LocationID' THEN 10
        WHEN 'Orphan pickup IDs' THEN 11
        WHEN 'Orphan dropoff IDs' THEN 12
    END